# Correlación y causalidad — Ejemplos

**Módulo 3 — Modelado y evaluación de enfoques no supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`3.1_Correlacion_y_Causalidad.pdf`](3.1_Correlacion_y_Causalidad.pdf) y lleva a código lo que allí se explica, usando un dataset real y muy popular: el **TMDB 5000 Movie Dataset** de Kaggle (información de más de 4.800 películas).

## Contenido

1. **Exploración visual**: cómo se ven una correlación positiva, una negativa y la ausencia de asociación lineal.
2. **El coeficiente de correlación de Pearson**: calcularlo e interpretarlo (signo e intensidad) sobre pares de variables reales de películas.
3. **Un coeficiente necesita un gráfico**: el famoso *cuarteto de Anscombe*, cuatro conjuntos de datos con el mismo coeficiente pero formas completamente distintas.
4. **Transformaciones logarítmicas** que revelan relaciones ocultas por la escala original.
5. **Mapas de correlación**: una matriz de calor con varias variables de las películas a la vez.
6. **Correlación ≠ causalidad**: el clásico ejemplo de los helados y los ahogamientos, con una variable de confusión — y cómo detectarla con código (correlación parcial).

> 💡 La sección 1 en adelante usa datos reales de Kaggle. La sección 2 en particular **requiere una cuenta de Kaggle y una clave de API** (gratis) — el proceso está explicado paso a paso en el notebook `1.6` del Módulo 1 y se resume aquí también. Las secciones 1 (patrones sintéticos), 3 (Anscombe) y 6 (helados y ahogamientos) usan datos generados en el propio notebook, así que no requieren descargar nada.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (7, 5)
sns.set_style('whitegrid')
rng = np.random.default_rng(42)  # semilla fija para que los ejemplos sean reproducibles

print('Librerías cargadas correctamente ✅')
print('pandas', pd.__version__, '· scipy', __import__('scipy').__version__)

---
## 1. Exploración visual: ¿qué relación muestran los datos?

Antes de calcular cualquier número, como en la diapositiva 3, generamos tres ejemplos sintéticos que ilustran los tres patrones básicos: **correlación positiva**, **correlación negativa** y **sin asociación lineal**.

In [ ]:
n = 60
x = rng.uniform(0, 10, n)

y_positiva = 2 * x + rng.normal(0, 2, n)
y_negativa = -2 * x + rng.normal(0, 2, n) + 40
y_sin_asociacion = rng.uniform(0, 20, n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(x, y_positiva, color='#1565c0', alpha=0.7)
axes[0].set_title(f'Correlación positiva (r = {np.corrcoef(x, y_positiva)[0,1]:.2f})')

axes[1].scatter(x, y_negativa, color='#c62828', alpha=0.7)
axes[1].set_title(f'Correlación negativa (r = {np.corrcoef(x, y_negativa)[0,1]:.2f})')

axes[2].scatter(x, y_sin_asociacion, color='#616161', alpha=0.7)
axes[2].set_title(f'Sin asociación lineal (r = {np.corrcoef(x, y_sin_asociacion)[0,1]:.2f})')

for ax in axes:
    ax.set_xlabel('x')
plt.tight_layout()
plt.show()

print('Fíjate en la dirección de la nube de puntos: hacia arriba-derecha (positiva),')
print('hacia abajo-derecha (negativa), o sin ninguna tendencia clara (sin asociación).')

---
## 2. El coeficiente de correlación de Pearson

Cargamos el dataset de películas y calculamos el coeficiente de Pearson para algunos pares de variables, usando la escala de interpretación de la diapositiva 4 (nula / débil / moderada / fuerte, positiva o negativa).

In [ ]:
import kagglehub
import os

try:
    ruta_dataset = kagglehub.dataset_download('tmdb/tmdb-movie-metadata')
    print('Dataset descargado en:', ruta_dataset)
except Exception as error:
    print('❌ No se pudo descargar el dataset de Kaggle.')
    print('Verifica tu conexión a internet y que tu API key esté configurada (ver el notebook 1.6 del Módulo 1).')
    print('Detalle del error:', error)
    raise

# Nos interesa específicamente el archivo con la información general de las películas
# (el dataset trae también un archivo de créditos/reparto que no usamos aquí).
archivos_csv = [f for f in os.listdir(ruta_dataset) if 'movies' in f.lower() and f.endswith('.csv')]
print('Archivo encontrado:', archivos_csv)

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])
peliculas = pd.read_csv(ruta_csv)

print(f'\nDataset cargado: {peliculas.shape[0]} filas x {peliculas.shape[1]} columnas')
peliculas[['title', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'runtime']].head()

In [ ]:
def interpretar_r(r):
    """Clasifica un coeficiente de Pearson según la escala de la diapositiva 4."""
    signo = 'positiva' if r > 0 else 'negativa' if r < 0 else 'nula'
    magnitud = abs(r)
    if magnitud < 0.1:
        return 'prácticamente nula'
    intensidad = 'débil' if magnitud < 0.3 else 'moderada' if magnitud < 0.5 else 'fuerte'
    return f'{intensidad} {signo}'

# Nota: 'budget' (presupuesto) y 'revenue' (recaudación) traen muchos ceros que en
# realidad son datos faltantes sin reportar, no presupuestos/recaudaciones de $0.
# Los excluimos para no distorsionar el coeficiente.
pares = [
    ('budget', 'revenue'),
    ('runtime', 'vote_average'),
    ('popularity', 'vote_count'),
]

for col_a, col_b in pares:
    sub = peliculas[[col_a, col_b]].replace(0, np.nan).dropna()
    r, valor_p = stats.pearsonr(sub[col_a], sub[col_b])
    print(f'{col_a} vs {col_b}: r = {r:.3f}  →  correlación {interpretar_r(r)}  (n = {len(sub)})')

**Lectura** (diapositiva 4): el presupuesto y la recaudación tienen una correlación fuerte y positiva —tiene sentido, las películas más costosas suelen recaudar más—, mientras que la duración explica bastante menos de la calificación promedio (correlación apenas moderada).

---
## 3. Un coeficiente necesita un gráfico: el cuarteto de Anscombe

Como advierte la diapositiva 5, **el coeficiente resume una sola característica de la relación**. El estadístico Francis Anscombe construyó en 1973 cuatro conjuntos de datos con **la misma media, la misma varianza y el mismo coeficiente de correlación** — pero que se ven completamente distintos al graficarlos. Los valores están tan bien documentados que los escribimos directamente en el notebook (no requieren descargar nada).

In [ ]:
anscombe = {
    'I':   {'x': [10,8,13,9,11,14,6,4,12,7,5],
            'y': [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]},
    'II':  {'x': [10,8,13,9,11,14,6,4,12,7,5],
            'y': [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]},
    'III': {'x': [10,8,13,9,11,14,6,4,12,7,5],
            'y': [7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]},
    'IV':  {'x': [8,8,8,8,8,8,8,19,8,8,8],
            'y': [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]},
}

fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for ax, (nombre, datos) in zip(axes.flat, anscombe.items()):
    x, y = np.array(datos['x']), np.array(datos['y'])
    r = np.corrcoef(x, y)[0, 1]

    ax.scatter(x, y, color='#6a1b9a', s=50)
    # línea de tendencia, solo para visualizar
    b, a = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, b * xs + a, color='#6a1b9a', alpha=0.4)

    ax.set_title(f'Conjunto {nombre}: media_x={x.mean():.1f}, media_y={y.mean():.2f}, r={r:.3f}')

plt.tight_layout()
plt.show()

print('Los cuatro conjuntos tienen prácticamente el mismo coeficiente (r ≈ 0.816), pero:')
print('  I:   una relación lineal razonable, con dispersión normal.')
print('  II:  una relación claramente CURVA, no lineal — el coeficiente no la capta.')
print('  III: una relación lineal casi perfecta, distorsionada por UN solo valor atípico.')
print('  IV:  casi todos los puntos comparten el mismo x=8; un único punto (AGRUPAMIENTO)')
print('       genera por sí solo todo el coeficiente.')

Esto es exactamente lo que dice la diapositiva 5: **curvaturas, agrupaciones y observaciones alejadas** pueden producir el mismo coeficiente que una relación lineal genuina. Por eso siempre hay que mirar el gráfico de dispersión, nunca confiar solo en el número.

---
## 4. Transformaciones que revelan relaciones

Como en la diapositiva 6, aplicamos logaritmos a una variable con una relación muy asimétrica: comparamos `popularity` y `vote_count` en su escala original y en escala logarítmica.

In [ ]:
sub = peliculas[(peliculas['popularity'] > 0) & (peliculas['vote_count'] > 0)]

r_original, _ = stats.pearsonr(sub['popularity'], sub['vote_count'])
r_log, _ = stats.pearsonr(np.log10(sub['popularity']), np.log10(sub['vote_count']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(sub['popularity'], sub['vote_count'], alpha=0.3, s=12, color='#00695c')
axes[0].set_xlabel('popularity (escala original)')
axes[0].set_ylabel('vote_count (escala original)')
axes[0].set_title(f'Escala original (r = {r_original:.3f})')

axes[1].scatter(np.log10(sub['popularity']), np.log10(sub['vote_count']), alpha=0.3, s=12, color='#00695c')
axes[1].set_xlabel('log10(popularity)')
axes[1].set_ylabel('log10(vote_count)')
axes[1].set_title(f'Escala logarítmica (r = {r_log:.3f})')

plt.tight_layout()
plt.show()

print(f'r en escala original: {r_original:.3f}')
print(f'r en escala logarítmica: {r_log:.3f}')
print('\nEn la escala original, la inmensa mayoría de películas se amontona cerca del origen')
print('y unos pocos éxitos gigantes estiran el eje — la tendencia real queda oculta.')
print('En escala logarítmica los puntos se reparten mejor y la relación lineal se ve mucho más clara.')

⚠️ Recuerda las tres advertencias de la diapositiva 6: el logaritmo **requiere valores estrictamente positivos** (por eso filtramos los ceros), cambia la interpretación de la escala (ahora hablamos de órdenes de magnitud, no de unidades), y **encontrar una relación más clara sigue sin demostrar causalidad**.

---
## 5. Mapas de correlación

Como en la diapositiva 7, construimos una matriz de calor con varias variables numéricas de las películas a la vez, para detectar de un vistazo qué pares están más y menos asociados.

In [ ]:
columnas_numericas = ['budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'runtime']
correlaciones = peliculas[columnas_numericas].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(correlaciones, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, ax=ax)
ax.set_title('Mapa de correlación — variables numéricas de las películas')
plt.tight_layout()
plt.show()

print('Identificar variables: revenue y vote_count están fuertemente correlacionadas (posible redundancia).')
print('vote_average es la variable que menos se relaciona linealmente con las demás.')
print('Contexto de tarea: si el objetivo fuera predecir vote_average, budget y revenue aportarían poco por sí solos.')

---
## 6. Correlación ≠ causalidad: helados y ahogamientos

Reproducimos el ejemplo clásico de las diapositivas 8 y 9 con datos sintéticos: generamos una variable de **confusión** (`temperatura`) que influye tanto en el consumo de helados como en los ahogamientos, sin que exista ninguna relación causal directa entre estos dos últimos.

In [ ]:
n2 = 200
temperatura = rng.normal(25, 6, n2)                              # variable de confusión
helados = 5 * temperatura + rng.normal(0, 15, n2)                  # depende de la temperatura
ahogamientos = 0.8 * temperatura + rng.normal(0, 3, n2)             # también depende de la temperatura

r_directa, _ = stats.pearsonr(helados, ahogamientos)
print(f'Correlación directa entre helados y ahogamientos: r = {r_directa:.3f}')
print('¡Una correlación fuerte, a pesar de que comer helados no provoca ahogamientos!')

fig, ax = plt.subplots()
sc = ax.scatter(helados, ahogamientos, c=temperatura, cmap='coolwarm')
ax.set_xlabel('Venta de helados')
ax.set_ylabel('Ahogamientos')
ax.set_title('Helados vs. ahogamientos (color = temperatura)')
plt.colorbar(sc, label='Temperatura (°C)')
plt.tight_layout()
plt.show()

print('\nEl color revela lo que el diagrama sin colorear escondía: los días más cálidos')
print('(en rojo) concentran tanto las ventas altas de helados como los ahogamientos.')

### ¿Cómo confirmamos que es la temperatura la que genera la asociación?

Calculamos la **correlación parcial**: le "restamos" a cada variable lo que se explica por la temperatura (los residuos de una regresión simple) y correlacionamos lo que queda. Si la asociación desaparece, confirma que la temperatura era la explicación.

In [ ]:
def residuos_de_regresion(y, x):
    """Devuelve lo que le sobra a 'y' después de descontar su relación lineal con 'x'."""
    pendiente, intercepto = np.polyfit(x, y, 1)
    return y - (pendiente * x + intercepto)

residuos_helados = residuos_de_regresion(helados, temperatura)
residuos_ahogamientos = residuos_de_regresion(ahogamientos, temperatura)

r_parcial, _ = stats.pearsonr(residuos_helados, residuos_ahogamientos)
print(f'Correlación directa (sin controlar):        r = {r_directa:.3f}')
print(f'Correlación parcial (controlando temperatura): r = {r_parcial:.3f}')
print('\nAl controlar por la variable de confusión, la correlación cae a prácticamente cero:')
print('la asociación entre helados y ahogamientos era enteramente explicada por la temperatura.')

---
## 7. Para pensar y discutir en clase

Elige cualquier par de variables del dataset de películas (puede ser uno de los que ya usamos u otro distinto) y sigue el esquema de la diapositiva 10 — **Observar → Medir → Contextualizar**:

1. **Observar**: haz un diagrama de dispersión. ¿Hay una tendencia lineal, una curva, agrupaciones o valores atípicos?
2. **Medir**: calcula el coeficiente de Pearson. ¿Qué tan fuerte e intensa es la asociación según la escala de la diapositiva 4?
3. **Contextualizar**: ¿tiene sentido pensar en una relación causal entre esas dos variables? Si la respuesta es sí, ¿en qué dirección iría la causalidad? Si la respuesta es no (o no estás seguro), ¿se te ocurre una posible variable de confusión, como la temperatura en el ejemplo de los helados?

No hay una única respuesta "correcta": lo importante es justificar cada paso con evidencia del propio dataset.

---
## Cierre

En este notebook llevamos a código las ideas de las diapositivas [`3.1_Correlacion_y_Causalidad.pdf`](3.1_Correlacion_y_Causalidad.pdf):

- Cómo se ven una **correlación positiva**, una **negativa** y la **ausencia de asociación lineal**.
- Cómo calcular e **interpretar el coeficiente de Pearson** (signo e intensidad).
- Por qué **un coeficiente necesita un gráfico** — el cuarteto de Anscombe como advertencia definitiva.
- Cómo una **transformación logarítmica** puede revelar una relación oculta por la escala original.
- Cómo leer un **mapa de correlación** con varias variables a la vez.
- Por qué **correlación no implica causalidad**, y cómo una **variable de confusión** puede generar una asociación aparente — y cómo detectarla calculando una correlación parcial.

### Recursos adicionales
- [TMDB 5000 Movie Dataset en Kaggle](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)
- [El cuarteto de Anscombe (Wikipedia)](https://es.wikipedia.org/wiki/Cuarteto_de_Anscombe)
- [Documentación de `scipy.stats.pearsonr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html)
- [Documentación de `seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)